# Exercise 5.1: Global Evaluation

Generate `dev.txt` and `test.txt` prediction files for the official
SemEval dev set and test set using the best model from Experiment L
(RoBERTa-large verbalizer).

**Output format:** One prediction per line (`0` = No PCL, `1` = PCL).

In [1]:
import os
import sys
import json
import logging

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from sklearn.metrics import classification_report

sys.path.insert(0, ".")
from utils.data import load_data, clean_text
from utils.pcl_dataset import PCLVerbalizerDataset
from utils.pcl_deberta import PCLVerbalizer
from utils.eval import evaluate

SEED = 42
DATA_DIR = "data"
OUT_DIR = "exp/out"
MODEL_OUT_DIR = OUT_DIR
EXP_NAME = "L_verbalizer"
MODEL_NAME = "roberta-large"
MAX_LENGTH = 256
BATCH_SIZE = 24
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s:\t%(message)s")
LOG = logging.getLogger(__name__)
LOG.info(f"Device: {DEVICE}")

login = None
try:
    login = os.getlogin()
except Exception as e:
    LOG.warning(f"Could not get login name: {e}")
if login == "jc4423":
    LOG.info("Running on lab machines, changing caches and model out dir")
    os.environ["HF_HOME"] = "/vol/bitbucket/jc4423/.cache/huggingface"
    os.environ["TRANSFORMERS_CACHE"] = "/vol/bitbucket/jc4423/.cache/huggingface"
    os.environ["HF_DATASETS_CACHE"] = "/vol/bitbucket/jc4423/.cache/huggingface"
    os.environ["PIP_CACHE_DIR"] = "/vol/bitbucket/jc4423/.cache/pip"
    os.environ["XDG_CACHE_HOME"] = "/vol/bitbucket/jc4423/.cache"
    MODEL_OUT_DIR = "/vol/bitbucket/jc4423/models/"

2026-03-04 14:19:05,442 INFO:	Device: cuda
2026-03-04 14:19:05,443 WARNING:	Could not get login name: [Errno 6] No such device or address


## 1. Load Best Model Config

Read the saved hyperparameters from the best Optuna trial.

In [2]:
with open(os.path.join(OUT_DIR, f"exp_{EXP_NAME}_best_params.json")) as f:
    best_config = json.load(f)

print(json.dumps(best_config, indent=2))

# Extract key params
BEST_THRESHOLD = best_config["best_threshold"]
TEMPLATE_IDX = best_config["template_idx"]
VERBALIZER_NAME = best_config["verbalizer"]

LOG.info(f"Best threshold: {BEST_THRESHOLD}")
LOG.info(f"Verbalizer: {VERBALIZER_NAME}, Template idx: {TEMPLATE_IDX}")

2026-03-04 14:19:05,507 INFO:	Best threshold: 0.275
2026-03-04 14:19:05,508 INFO:	Verbalizer: true_false, Template idx: 2


{
  "lr": 6.977921490278437e-06,
  "weight_decay": 0.000691662498760998,
  "head_lr_multiplier": 10,
  "verbalizer": "true_false",
  "template_idx": 2,
  "batch_size": 32,
  "num_epochs": 12,
  "patience": 4,
  "best_threshold": 0.275,
  "warmup_fraction": 0.06,
  "label_smoothing": 0.0
}


## 2. Setup Verbalizer & Tokeniser

In [3]:
tokeniser = AutoTokenizer.from_pretrained(MODEL_NAME)

# Verbalizer sets (must match training)
VERBALIZER_SETS = [
    ("Yes_No",     ["Yes"],  ["No"]),
    ("yes_no",     ["yes"],  ["no"]),
    ("True_False", ["True"], ["False"]),
    ("true_false", ["true"], ["false"]),
]

TEMPLATE_OPTIONS = [
    'Is the following text patronising or condescending? "{text}" Answer: {mask}',
    'Does the author of this text sound patronising or condescending? "{text}" Answer: {mask}',
    'Is this text talking down to people? "{text}" Answer: {mask}',
    'Does the following text contain patronising language? "{text}" Answer: {mask}',
]


def words_to_first_subword_ids(words: list[str]) -> list[int]:
    return [tokeniser.encode(w, add_special_tokens=False)[0] for w in words]


# Resolve verbalizer IDs and template
verb_set = [v for v in VERBALIZER_SETS if v[0] == VERBALIZER_NAME][0]
pos_ids = words_to_first_subword_ids(verb_set[1])
neg_ids = words_to_first_subword_ids(verb_set[2])
template = TEMPLATE_OPTIONS[TEMPLATE_IDX]

LOG.info(f"Verbalizer '{VERBALIZER_NAME}': pos_ids={pos_ids}, neg_ids={neg_ids}")
LOG.info(f"Template: {template}")

2026-03-04 14:19:05,966 INFO:	HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
2026-03-04 14:19:06,103 INFO:	HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-04 14:19:06,202 INFO:	HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-04 14:19:06,296 INFO:	HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-03-04 14:19:06,393 INFO:	HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-04 14:19:06,492 INFO:	HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-large/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-03-0

## 3. Load Best Model Weights

In [4]:
model = PCLVerbalizer(
    pos_verbalizer_ids=pos_ids,
    neg_verbalizer_ids=neg_ids,
    model_name=MODEL_NAME,
    gradient_checkpointing=False,
    cache_dir="/vol/bitbucket/jc4423/.cache/huggingface" if login == "jc4423" else None,
).to(DEVICE)

state_dict = torch.load(
    os.path.join(MODEL_OUT_DIR, f"exp_{EXP_NAME}_best_model.pt"),
    map_location=DEVICE,
)
model.load_state_dict(state_dict)
model.eval()
LOG.info("Best model loaded.")

2026-03-04 14:19:06,941 INFO:	HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: roberta-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-04 14:19:11,521 INFO:	Verbalizer model loaded: RobertaForMaskedLM, n_pos=1, n_neg=1, pos_ids=[29225], neg_ids=[22303]
2026-03-04 14:19:11,522 INFO:	  backbone attr: .None, head attr: .None
2026-03-04 14:19:20,650 INFO:	Best model loaded.


## 4. Generate Dev Predictions (`dev.txt`)

Use the official dev set (labels available) — also serves as a sanity check.

In [5]:
# Load dev set manually (no dropna — keep all rows so prediction count matches)
col_names = ["par_id", "art_id", "keyword", "country_code", "text", "label"]
df = pd.read_csv(
    os.path.join(DATA_DIR, "dontpatronizeme_pcl.tsv"),
    sep="\t", skiprows=4, names=col_names, index_col="par_id",
)
df["binary_label"] = (df["label"] >= 2).astype(int)
df["text"] = df["text"].fillna("").apply(clean_text)

dev_ids = pd.read_csv(os.path.join(DATA_DIR, "dev_semeval_parids-labels.csv"))["par_id"].values
print(f"Order of dev_ids: {dev_ids[:10]}")
dev_df = df.loc[df.index.isin(dev_ids), ["text", "binary_label"]].copy()
LOG.info(f"Dev set: {len(dev_df)} samples, {dev_df['binary_label'].sum()} positive")

dev_ds = PCLVerbalizerDataset(
    texts=dev_df["text"].tolist(),
    labels=dev_df["binary_label"].tolist(),
    max_length=MAX_LENGTH,
    tokeniser=tokeniser,
    template=template,
)
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False)

dev_metrics = evaluate(model, DEVICE, dev_loader, threshold=BEST_THRESHOLD)

print(f"Dev F1: {dev_metrics['f1']:.4f}")
print(f"Dev Precision: {dev_metrics['precision']:.4f}")
print(f"Dev Recall: {dev_metrics['recall']:.4f}")
print()
print(classification_report(
    dev_metrics["labels"], dev_metrics["preds"],
    target_names=["Non-PCL", "PCL"],
))

# Write dev.txt
dev_preds = dev_metrics["preds"]
with open("dev.txt", "w") as f:
    for p in dev_preds:
        f.write(f"{p}\n")

LOG.info(f"dev.txt written: {len(dev_preds)} predictions")
assert len(dev_preds) == len(dev_df), f"Expected {len(dev_df)} but got {len(dev_preds)}"

2026-03-04 14:19:20,869 INFO:	Dev set: 2094 samples, 199 positive


Order of dev_ids: [4046 1279 8330 4063 4089  432 4177 3963 2001  369]


2026-03-04 14:19:21,489 WARNING:	Sample 1483: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 14:20:16,424 INFO:	dev.txt written: 2094 predictions


Dev F1: 0.5919
Dev Precision: 0.5344
Dev Recall: 0.6633

              precision    recall  f1-score   support

     Non-PCL       0.96      0.94      0.95      1895
         PCL       0.53      0.66      0.59       199

    accuracy                           0.91      2094
   macro avg       0.75      0.80      0.77      2094
weighted avg       0.92      0.91      0.92      2094



## 5. Generate Test Predictions (`test.txt`)

Load the unlabelled test set (`task4_test.tsv`) and generate predictions.

In [6]:
# Load test set (no labels)
test_df = pd.read_csv(
    os.path.join(DATA_DIR, "task4_test.tsv"),
    sep="\t",
    header=None,
    names=["par_id", "art_id", "keyword", "country_code", "text"],
    index_col="par_id",
)
test_df["text"] = test_df["text"].apply(clean_text)
LOG.info(f"Test set: {len(test_df)} samples")
test_df.head()

2026-03-04 14:20:16,553 INFO:	Test set: 3832 samples


,art_id,keyword,country_code,text
par_id,,,,
t_0,@@7258997,vulnerable,us,"In the meantime , conservatives are working to..."
t_1,@@16397324,women,pk,In most poor households with no education chil...
t_2,@@16257812,migrant,ca,The real question is not whether immigration i...
t_3,@@3509652,migrant,gb,"In total , the country 's immigrant population..."
t_4,@@477506,vulnerable,ca,"Members of the church , which is part of Ken C..."


In [7]:
# Create test dataset (no labels)
test_ds = PCLVerbalizerDataset(
    texts=test_df["text"].tolist(),
    labels=None,
    max_length=MAX_LENGTH,
    tokeniser=tokeniser,
    template=template,
)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# Run inference
model.eval()
all_scores = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        mask_token_indices = batch["mask_token_indices"].to(DEVICE)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            mask_token_indices=mask_token_indices,
        ).squeeze(-1)
        all_scores.append(logits.cpu())

all_scores = torch.cat(all_scores)
probs = torch.sigmoid(all_scores)
test_preds = (probs >= BEST_THRESHOLD).long().numpy()

LOG.info(f"Test predictions: {len(test_preds)} total, "
         f"{test_preds.sum()} positive ({test_preds.mean():.2%})")

# Write test.txt
with open("test.txt", "w") as f:
    for p in test_preds:
        f.write(f"{p}\n")

LOG.info(f"test.txt written: {len(test_preds)} predictions")
assert len(test_preds) == 3832, f"Expected 3832 but got {len(test_preds)}"

2026-03-04 14:20:17,928 WARNING:	Sample 1847: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 14:20:17,934 WARNING:	Sample 2164: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 14:20:17,951 WARNING:	Sample 3185: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 14:20:17,959 WARNING:	Sample 3623: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 14:21:58,124 INFO:	Test predictions: 3832 total, 414 positive (10.80%)
2026-03-04 14:21:58,146 INFO:	test.txt written: 3832 predictions


## 6. Sanity Checks

In [8]:
# Verify files
for fname, expected_len in [("dev.txt", len(dev_df)), ("test.txt", 3832)]:
    with open(fname) as f:
        lines = f.readlines()
    n_lines = len(lines)
    values = set(line.strip() for line in lines)
    print(f"{fname}: {n_lines} lines, values: {values}")
    assert n_lines == expected_len, f"{fname}: expected {expected_len} lines, got {n_lines}"
    assert values <= {"0", "1"}, f"{fname}: unexpected values {values - {'0', '1'}}"

print("\nAll checks passed!")

dev.txt: 2094 lines, values: {'1', '0'}
test.txt: 3832 lines, values: {'1', '0'}

All checks passed!
